# ALS Baseline: Books + Movies Intersection

Train one implicit ALS model on merged Books and Movies interactions from the prepared Google Drive splits. Recommendations are filtered back to the target domain before MAP@10 evaluation.

In [ ]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout als && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

import implicit
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

from src.evaluation.metrics import map_at_k

## Load Prepared Splits

The split CSVs are expected to contain the Books/Movies intersecting users and time-based train/validation/test rows.

In [ ]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [ ]:
def add_domain(df, domain):
    df = df.copy()
    df['domain'] = domain
    df['item_id'] = domain + '::' + df['parent_asin'].astype(str)
    return df

books_train = add_domain(books_train, 'books')
books_valid = add_domain(books_valid, 'books')
books_test = add_domain(books_test, 'books')

movies_train = add_domain(movies_train, 'movies')
movies_valid = add_domain(movies_valid, 'movies')
movies_test = add_domain(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df = pd.concat([books_test, movies_test], ignore_index=True)

train_df[['user_id', 'item_id', 'domain']].head()

## Shared Indexes and Train Matrix

In [ ]:
all_users = train_df['user_id'].drop_duplicates().to_numpy()
all_items = train_df['item_id'].drop_duplicates().to_numpy()

user2idx = {user_id: idx for idx, user_id in enumerate(all_users)}
item2idx = {item_id: idx for idx, item_id in enumerate(all_items)}
idx2item = {idx: item_id for item_id, idx in item2idx.items()}
item_domain = {item2idx[item_id]: item_id.split('::', 1)[0] for item_id in all_items}

train_df['uidx'] = train_df['user_id'].map(user2idx)
train_df['iidx'] = train_df['item_id'].map(item2idx)

n_users = len(user2idx)
n_items = len(item2idx)

values = np.ones(len(train_df), dtype=np.float32)
user_item_train = csr_matrix(
    (values, (train_df['uidx'].to_numpy(), train_df['iidx'].to_numpy())),
    shape=(n_users, n_items),
)
user_item_train.sum_duplicates()

print(f'users={n_users:,}, items={n_items:,}, train_interactions={user_item_train.nnz:,}')
print(train_df['domain'].value_counts())

In [ ]:
train_seen_idx_by_user = {
    user_idx: set(user_item_train[user_idx].indices)
    for user_idx in range(user_item_train.shape[0])
}

domain_item_indices = {
    domain: np.array([idx for idx, item_dom in item_domain.items() if item_dom == domain], dtype=np.int64)
    for domain in ['books', 'movies']
}

assert user_item_train.shape == (n_users, n_items)
assert len(domain_item_indices['books']) > 0
assert len(domain_item_indices['movies']) > 0

## Train Collective ALS

In [ ]:
FACTORS = 64
REGULARIZATION = 0.01
ITERATIONS = 20
ALPHA = 40
K = 10

als_model = implicit.als.AlternatingLeastSquares(
    factors=FACTORS,
    regularization=REGULARIZATION,
    iterations=ITERATIONS,
    random_state=42,
)

item_user_train = (user_item_train * ALPHA).T.tocsr()
als_model.fit(item_user_train)

## Domain-Filtered Recommendations

In [ ]:
def relevant_items_by_user(split_df, target_domain):
    domain_rows = split_df[split_df['domain'] == target_domain]
    return domain_rows.groupby('user_id')['item_id'].agg(set).to_dict()


def recommend_for_users(user_ids, target_domain, k=10):
    candidates = domain_item_indices[target_domain]
    candidate_factors = als_model.item_factors[candidates]
    recommendations = {}

    for user_id in user_ids:
        user_idx = user2idx.get(user_id)
        if user_idx is None:
            recommendations[user_id] = []
            continue

        scores = candidate_factors @ als_model.user_factors[user_idx]
        scores = scores.astype(np.float64, copy=True)

        seen_items = train_seen_idx_by_user.get(user_idx, set())
        if seen_items:
            seen_mask = np.isin(candidates, list(seen_items))
            scores[seen_mask] = -np.inf

        finite_count = np.isfinite(scores).sum()
        if finite_count == 0:
            recommendations[user_id] = []
            continue

        top_n = min(k, int(finite_count))
        top_positions = np.argpartition(-scores, top_n - 1)[:top_n]
        top_positions = top_positions[np.argsort(-scores[top_positions])]
        recommendations[user_id] = [idx2item[int(candidates[pos])] for pos in top_positions]

    return recommendations


def evaluate_split(split_df, target_domain, k=10):
    relevant = relevant_items_by_user(split_df, target_domain)
    recommended = recommend_for_users(relevant.keys(), target_domain, k=k)
    return map_at_k(recommended, relevant, k=k), recommended, relevant

## Smoke Checks

In [ ]:
for domain in ['books', 'movies']:
    sample_relevant_all = relevant_items_by_user(valid_df, domain)
    sample_users = list(sample_relevant_all.keys())[:50]
    sample_recs = recommend_for_users(sample_users, domain, k=K)

    assert all(item.startswith(f'{domain}::') for recs in sample_recs.values() for item in recs)
    for user_id, recs in sample_recs.items():
        user_idx = user2idx[user_id]
        seen_item_ids = {idx2item[idx] for idx in train_seen_idx_by_user[user_idx]}
        assert not (set(recs) & seen_item_ids)

    sample_relevant = {user_id: sample_relevant_all[user_id] for user_id in sample_users}
    assert isinstance(map_at_k(sample_recs, sample_relevant, k=K), float)

## Evaluate and Save Metrics

In [ ]:
books_valid_map10, books_valid_recs, books_valid_relevant = evaluate_split(valid_df, 'books', k=K)
movies_valid_map10, movies_valid_recs, movies_valid_relevant = evaluate_split(valid_df, 'movies', k=K)
books_test_map10, books_test_recs, books_test_relevant = evaluate_split(test_df, 'books', k=K)
movies_test_map10, movies_test_recs, movies_test_relevant = evaluate_split(test_df, 'movies', k=K)

metrics = {
    'books_valid_map@10': books_valid_map10,
    'movies_valid_map@10': movies_valid_map10,
    'valid_macro_map@10': (books_valid_map10 + movies_valid_map10) / 2,
    'books_test_map@10': books_test_map10,
    'movies_test_map@10': movies_test_map10,
    'test_macro_map@10': (books_test_map10 + movies_test_map10) / 2,
}

metrics

In [ ]:
os.makedirs('results', exist_ok=True)

result_row = {
    'model': 'implicit_als_books_movies_joint_binary',
    'factors': FACTORS,
    'regularization': REGULARIZATION,
    'iterations': ITERATIONS,
    'alpha': ALPHA,
    'k': K,
    'n_users': n_users,
    'n_items': n_items,
    'n_train_interactions': user_item_train.nnz,
    **metrics,
}

results_df = pd.DataFrame([result_row])
results_df.to_csv('results/als_baseline_metrics.csv', index=False)
results_df